In [ ]:
## Python (jupyter_env)
### Run in this environment

import pyemma
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA, IncrementalPCA
import mdtraj as md
import pandas as pd
from rdkit import Chem
import gc
import time
import glob
import os

plt.rcParams['font.family']  'serif'
plt.rcParams['font.serif']  ['Times New Roman'] + plt.rcParams['font.serif']

%matplotlib inline
%load_ext memory_profiler

gc.collect()

In [ ]:
gc.collect()
#  Ligand info with fragment SMILES 
ligand_info  pd.DataFrame([
    {
        "name": "LIG1",
        "traj_file": "traj_file.dcd",
        "top_file": "top_file.prmtop",
        "fragment_smiles": "C3C4CCNC4NCN3",
    },
        {
        "name": "LIG2",
        "traj_file": "trj_file.dcd",
        "top_file": "top_file.prmtop",
        "fragment_smiles": "C3C4CCNC4NCN3",
    },
    {
        "name": "LIG3",
        "traj_file": "traj_file.dcd",
        "top_file": "top_file.prmtop",
        "fragment_smiles": "C3C4CCNC4NCN3",
    },
])

gc.collect()

In [ ]:
cutoff  0.5  # nm
n_top  20
min_contact_fraction  0.05
# Chunk size for distance calculation
chunk_size  500
pca_data  {}
results_summary  []
ref_fragment_coords  None  # For centroidbased alignment
# Helper: compute centroid of ligand fragment atoms
def fragment_centroid(traj, atom_indices):
    return traj.xyz[:, atom_indices, :].mean(axis1)
# Main loop
for idx, row in ligand_info.iterrows():
    name  row["name"]
    traj_file  row["traj_file"]
    top_file  row["top_file"]
    fragment_smiles  row["fragment_smiles"]
    print(f"\n Processing {name} ")
    total_start  time.time()
    # Load trajectory
    t0  time.time()
    traj  md.load(
        traj_file,
        toptop_file, stride100
    )
    top  traj.topology
    print(
        f"  Load trajectory: "
        f"{time.time()  t0:.2f} s"
    )
    print(
        f"  Frames: {traj.n_frames:,}"
    )
    # Identify ligand residue
    ligand_residues  [
        r for r in top.residues
        if name.upper() in r.name.upper()
    ]
    if not ligand_residues:
        print(
            f" Ligand '{name}' not found, skipping."
        )
        del traj
        del top
        gc.collect()
        continue
    ligand_res  ligand_residues[0] 
    # Ligand heavy atoms 
    ligand_heavy_atoms  np.asarray(
        [
            a.index
            for a in ligand_res.atoms
            if a.element.symbol ! "H"
        ],
        dtypenp.int32
    )
    print(
        f"  Ligand heavy atoms: "
        f"{len(ligand_heavy_atoms):,}"
    )
    # Compute ligand centroid
    t1  time.time()
    frag_cent  fragment_centroid(
        traj,
        ligand_heavy_atoms
    )
    print(
        f"  Centroid: "
        f"{time.time()  t1:.2f} s"
    )
    # Centroid alignment
    if ref_fragment_coords is None:
        ref_fragment_coords  frag_cent
        aligned_xyz  traj.xyz.copy()
        print(
            f" Using {name} fragment as reference "
            f"for centroid alignment."
        )
    else:
        translation  (
            ref_fragment_coords[0]
             frag_cent[0]
        )
        aligned_xyz  (
            traj.xyz
            + translation[None, None, :]
        )
    traj.xyz[:]  aligned_xyz
    # Free temporary arrays
    del aligned_xyz
    del frag_cent
    gc.collect()
    # Protein heavy atoms
    protein_atoms  top.select(
        "protein and not name H"
    )
    print(
        f"  Protein heavy atoms: "
        f"{len(protein_atoms):,}"
    )
    # Compute ligandprotein pairs
    t2  time.time()
    pairs  np.array(
        [
            [i, j]
            for i in ligand_heavy_atoms
            for j in protein_atoms
        ],
        dtypenp.int32
    )
    print(
        f"  Pair generation: "
        f"{time.time()  t2:.2f} s"
    )
    print(
        f"  Number of pairs: "
        f"{len(pairs):,}"
    )
    # Protein residue index corresponding to every pair
    protein_res_indices  np.asarray(
        [
            top.atom(j).residue.index
            for _, j in pairs
        ],
        dtypenp.int32
    )
    unique_residues  np.unique(
        protein_res_indices
    )
    # Contact count for every residue
    # Instead of storing a 50,000 × 100,000+ matrix,
    # process trajectory in chunks. 
    contact_counts  {
        r: 0
        for r in unique_residues
    }
    n_frames  traj.n_frames
    print(
        f"  Chunk size: {chunk_size}"
    )
    print(
        f"  Number of chunks: "
        f"{int(np.ceil(n_frames / chunk_size)):,}"
    )
    t_dist  time.time()
    # Chunk loop
    for start in range(
        0,
        n_frames,
        chunk_size
    ):
        end  min(
            start + chunk_size,
            n_frames
        )
        # Extract trajectory chunk
        traj_chunk  traj.slice(
            np.arange(start, end)
        )
        # 
        # Compute distances ONLY for this chunk
        # 
        distances  md.compute_distances(
            traj_chunk,
            pairs
        )
        # Contact map
        contact_map  (
            distances < cutoff
        ) 
        # Contact frequency per residue
        for r in unique_residues:
            mask  (
                protein_res_indices  r
            )
            contact_counts[r] + np.sum(
                contact_map[:, mask].any(axis1)
            )
        # Free chunk memory immediately
        del traj_chunk
        del distances
        del contact_map
        # Progress
        if (
            start  0
            or start % (chunk_size * 10)  0
        ):
            print(
                f"    Frames "
                f"{start:,}{end:,} / "
                f"{n_frames:,}"
            )
        gc.collect()
    # Distance calculation finished
    print(
        f"  Chunked distance/contact calculation: "
        f"{time.time()  t_dist:.2f} s"
    )
    # Contact frequency per residue
    res_contact_fraction  {}
    for r in unique_residues:
        freq  (
            contact_counts[r]
            / n_frames
        )
        if freq > min_contact_fraction:
            res_contact_fraction[r]  freq
    # Free contact calculation arrays
    del pairs
    del protein_res_indices
    del unique_residues
    del contact_counts
    gc.collect()
    # Check whether contacts exist
    if not res_contact_fraction:
        print(
            f" No residues contacted > "
            f"{min_contact_fraction:.2f}, skipping."
        )
        del traj
        del top
        gc.collect()
        continue
    # Top 20 residues
    top_residues  sorted(
        res_contact_fraction.items(),
        keylambda x: x[1],
        reverseTrue
    )[:n_top]
    print(
        f"  Top {len(top_residues)} contacted residues:"
    )
    for r, f in top_residues:
        res_obj  top.residue(r)
        print(
            f"    {res_obj.name:4s} "
            f"{res_obj.resSeq:5d} "
            f"{f:.3f}"
        )
        results_summary.append(
            {
                "Ligand": name,
                "ResName": res_obj.name,
                "ResSeq": res_obj.resSeq,
                "ContactFraction": f
            }
        )
    # Store Cα coordinates for PCA
    top_residue_indices  {
        r
        for r, _
        in top_residues
    }
    ca_indices  np.asarray(
        [
            a.index
            for a in top.atoms
            if (
                a.name  "CA"
                and
                a.residue.index
                in top_residue_indices
            )
        ],
        dtypenp.int32
    )
    del top_residue_indices
    if not len(ca_indices):
        print(
            f" No Cα atoms found for top residues, "
            f"skipping {name}."
        )
        del ca_indices
        del traj
        del top
        gc.collect()
        continue
    # Extract CA trajectory
    ca_traj  traj.atom_slice(
        ca_indices
    )
    xyz  ca_traj.xyz.reshape(
        traj.n_frames,
        1
    )
    # Store data
    pca_data[name]  {
        "traj": traj,
        "ca_indices": ca_indices,
        "xyz": xyz
    }
    # Free temporary object
    del ca_traj
    del xyz
    del res_contact_fraction
    del top_residues
    gc.collect()
    print(
        f" Finished {name}; "
        f"garbage collection completed."
    )
    print(
        f"  Total time for {name}: "
        f"{time.time()  total_start:.2f} s"
    )
# Save top contacts
# contact_df  pd.DataFrame(results_summary)
# contact_df.to_csv(
#     "top20_contacts.csv",
#     indexFalse
# )
# print(
#     "\n Top 20 residue contacts saved to "
#     "'top20_contacts.csv'."
# ) 
# PCA on union of all top residues
print(
    "\n Preparing PCA "
)
all_resseq  sorted(
    set(
        [
            r["ResSeq"]
            for r in results_summary
        ]
    )
)
print(
    f"Total unique residues for PCA: "
    f"{len(all_resseq)}"
)
traj_arrays  []
labels  []
# Prepare PCA arrays
for name, data in pca_data.items():
    print(
        f"  Preparing PCA data for {name}"
    )
    traj  data["traj"]
    xyz_frames  []
    # Build CA lookup once
    ca_by_resseq  {
        a.residue.resSeq: a.index
        for a in traj.topology.atoms
        if a.name  "CA"
    }
    # Extract coordinates for union of residues
    for resseq in all_resseq:
        ca_index  ca_by_resseq.get(
            resseq
        )
        if ca_index is not None:

            xyz_frames.append(
                traj.xyz[
                    :,
                    ca_index,
                    :
                ]
            )
        else:
            xyz_frames.append(
                np.zeros(
                    (
                        traj.n_frames,
                        3
                    ),
                    dtypetraj.xyz.dtype
                )
            )
    # Combine residue coordinates
    traj_array  np.hstack(
        xyz_frames
    )
    traj_arrays.append(
        traj_array
    )
    labels.extend(
        [name] * traj_array.shape[0]
    )
    print(
        f"    Shape: {traj_array.shape}"
    )
    # Free temporary objects
    del xyz_frames
    del ca_by_resseq
    del traj_array

    gc.collect()
# Combine all trajectories
print(
    "\nCombining PCA data..."
)
all_data  np.vstack(
    traj_arrays
)
print(
    f"Combined PCA data shape: "
    f"{all_data.shape}"
)
del traj_arrays
gc.collect()
# Run PCA
print(
    "\n Running PCA "
)
pca  IncrementalPCA(
    n_components2,
    batch_size100
)
# IMPORTANT:
# Do NOT run pca.fit() followed by fit_transform().
# fit_transform() performs the fit already.
t_pca  time.time()
%memit proj  pca.fit_transform(all_data)
print(
    f"PCA time: "
    f"{time.time()  t_pca:.2f} s"
)
# Free combined PCA data
del all_data
gc.collect()
# Split projections by ligand
proj_split  {}
start  0
for name, data in pca_data.items():
    n_frames  (
        data["traj"].n_frames
    )
    proj_split[name]  (
        proj[
            start:start + n_frames,
            :
        ]
    )
    start + n_frames

    gc.collect()
# Save PC1 and PC2 projections per ligand
for name, proj_data in proj_split.items():

    out_file  (
        f"{name}_PCA.dat"
    )

    np.savetxt(
        out_file,
        proj_data,
        fmt"%.6f",
        header"PC1\tPC2",
        comments""
    )

    print(
        f"Saved PCA projection for "
        f"{name} > {out_file}"
    )

    del proj_data

    gc.collect()
# Final cleanup
del proj
del proj_split
gc.collect()
print(
    "\n PCA analysis completed successfully "
)

In [ ]:
pca_files  sorted(glob.glob("*_PCA.dat"))

name_map  {"LIG1": "Name1", "LIG2": "Name2", "LIG3": "Name3"}
plot_order  ["LIG1", "LIG2", "LIG3"]
color_map  {"LIG1": "#0000ff", "LIG2": "#ffb214", "LIG3": "#6dfa61"}

fig, ax  plt.subplots(figsize(9, 7))

for name in plot_order:
    pca_file  f"{name}_PCA.dat"
    if not os.path.exists(pca_file):
        continue

    data  np.loadtxt(pca_file, skiprows1)
    ax.scatter(
        data[:, 0], data[:, 1],
        s8, alpha0.60,
        colorcolor_map.get(name),
        labelname_map.get(name),
        edgecolors"none"
    )

ax.set_xlabel("PC1", fontsize16)
ax.set_ylabel("PC2", fontsize16)

ax.legend(
    fontsize12, title_fontsize13,
    loc"upper right", markerscale2.5,
    handletextpad0.6, frameonTrue
)

ax.tick_params(axis"both", labelsize13)
ax.grid(False)
plt.tight_layout()
plt.savefig("PCA_combined4.png", dpi600, bbox_inches"tight")
plt.show()